# HippoVoice — Colab Runner (T4 edition)

**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

T4 has 16GB VRAM. Models are loaded selectively — never all at once.

| Section | Models loaded | VRAM used |
|---------|--------------|----------|
| 0. Setup | none | 0GB |
| 1. CPU tests | none | 0GB |
| 2. Benchmarks | Qwen3-4B 4-bit only | ~3GB |
| 3. Voice test | Canary-Qwen + Qwen3-4B 4-bit | ~8GB, then swap to TTS |

# Auto-fetch latest code + enable module auto-reload
# Re-run this single cell whenever you push a change — everything else stays live.

import os, sys

REPO_DIR = '/content/hippovoice'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull --quiet
else:
    !git clone https://github.com/shivansh193/hippovoice.git {REPO_DIR} --quiet

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Auto-reload: any imported module is reloaded before each cell runs.
# This means after a git pull, changes are live immediately — no runtime restart needed.
%load_ext autoreload
%autoreload 2

print(f'[{os.popen("git -C " + REPO_DIR + " log -1 --format=%h\\ %s").read().strip()}]')
print('Auto-reload active. Pull + re-run this cell anytime you push a change.')

In [ ]:
import numpy as np
print(f'numpy {np.__version__} — if this shows < 2.0, re-run the setup cell above and restart runtime.')

In [ ]:
# Fix numpy first — force a clean consistent 2.x install before anything else.
# Colab's base env needs numpy>=2.0; partial installs leave it in a broken state.
!pip install -q --force-reinstall "numpy==2.0.2"

# System libraries
!apt-get install -q -y portaudio19-dev libsndfile1

# Rust — needed to build tokenizers (fish-speech dep)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --quiet
import os; os.environ['PATH'] += ':/root/.cargo/bin'

# Core deps
!pip install -q chromadb networkx sentence-transformers soundfile jiwer fastapi pytest pytest-mock

# 4-bit quantization support
!pip install -q bitsandbytes accelerate

# NeMo for Canary-Qwen STT
!pip install -q Cython
!pip install -q nemo_toolkit[asr]

# Fish Speech TTS
!git clone -q https://github.com/fishaudio/fish-speech /content/fish-speech 2>/dev/null || git -C /content/fish-speech pull -q
!pip install -q -e /content/fish-speech --no-build-isolation 2>&1 | tail -5

import numpy as np
print(f'numpy {np.__version__}')
print('Done.')

In [ ]:
import os, sys

REPO_DIR = '/content/hippovoice'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/shivansh193/hippovoice.git {REPO_DIR}

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('Ready:', os.getcwd())
!ls

In [ ]:
!pytest tests/test_scorer.py tests/test_extractor.py tests/test_decay.py tests/test_context.py -v --tb=short

In [ ]:
# Verify salience math
from memory.scorer import compute_salience

checks = [
    ('fear fresh',      compute_salience(1.0, {'label': 'fear',    'intensity': 0.95}, 0,  0), '> 3.0'),
    ('neutral 45 turns',compute_salience(1.0, {'label': 'neutral', 'intensity': 0.01}, 0, 45), '< 0.25'),
    ('fear 45 turns',   compute_salience(1.0, {'label': 'fear',    'intensity': 0.90}, 0, 45), '> 0.25'),
]
for label, val, expectation in checks:
    print(f'{label:20s}: {val:.4f}  ({expectation})')

## 2. Memory Core + Store Tests

These need sentence-transformers (CPU, no GPU).

In [ ]:
!pytest tests/test_store.py tests/test_retriever.py -v --tb=short

In [ ]:
# Manual memory walkthrough to sanity-check the full stack without LLM
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve
import numpy as np

mem = HippoMemory()
entries = [
    ('My dog Max got diagnosed with cancer. I am devastated.',  'sadness', 0.9),
    ('The weather was cloudy today.',                           'neutral', 0.05),
    ('Max had his first chemo session. I held him the whole time.', 'sadness', 0.85),
    ('I had cereal for breakfast.',                             'neutral', 0.05),
]

for i, (content, label, intensity) in enumerate(entries):
    mem.add({'content': content,
             'emotion': {'label': label, 'intensity': intensity},
             'base_weight': 1.0, 'recall_count': 0, 'turn_created': i})

print(f'Stored {mem.count()} memories\n')
print('Query: "tell me about Max"')
results = hippo_retrieve('tell me about Max', mem, mem.graph, current_turn=4, top_k=4)
for r in results:
    print(f'  [{r["current_salience"]:.3f}] {r["content"]}')

## 3. Load LLM (Qwen3-4B, 4-bit — fits in ~3GB VRAM)

This is the only model needed for benchmarks.

In [ ]:
import torch
print(f'VRAM free before: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

from llm.client import LLMClient
# load_in_4bit=True (default) — Qwen3-4B in NF4 uses ~3GB
llm = LLMClient(model_name='Qwen/Qwen3-4B', load_in_4bit=True)

print(f'VRAM used after:  {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Smoke test
resp = llm.generate(
    system='Answer in one sentence.',
    messages=[{'role': 'user', 'content': 'What is 2 + 2?'}],
    max_tokens=30
)
print('Response:', resp)
assert '4' in resp
print('LLM working')

## 4. Signal/Noise Benchmark (Core Research Claim)

In [ ]:
from pipeline import HippoVoicePipeline
from baselines.naive_rag import NaiveRAG
from benchmarks.signal_noise.run import run_signal_noise_benchmark

print('Running signal/noise benchmark (~5 min)...\n')

hippo = HippoVoicePipeline(llm_client=llm, text_only=True)
naive = NaiveRAG()

hippo_result = run_signal_noise_benchmark(hippo, 'HippoVoice')
naive_result  = run_signal_noise_benchmark(naive,  'NaiveRAG')

print('=' * 50)
print(f'HippoVoice  noise rate: {hippo_result["noise_rate"]:.1%}  (signal={hippo_result["signal_count"]}, noise={hippo_result["noise_count"]})')
print(f'Naive RAG   noise rate: {naive_result["noise_rate"]:.1%}  (signal={naive_result["signal_count"]}, noise={naive_result["noise_count"]})')
print('=' * 50)
print(f'HippoVoice < 10% noise: {"PASS" if hippo_result["noise_rate"] < 0.10 else "FAIL"}')
print(f'Naive RAG  > 40% noise: {"PASS" if naive_result["noise_rate"]  > 0.40 else "FAIL"}')

## 5. LoCoMo Benchmark

~20 min on T4 with Qwen3-4B 4-bit.

In [ ]:
!pip install -q datasets

In [ ]:
from benchmarks.locomo.evaluate import run_locomo
from pipeline import HippoVoicePipeline

pipe = HippoVoicePipeline(llm_client=llm, text_only=True)
result = run_locomo(pipeline=pipe, num_conversations=30)

print(f'LoCoMo accuracy: {result["accuracy"]:.1%}  ({result["correct"]}/{result["total"]})')
print(f'Mem0 baseline:   ~65%')
print(f'Above baseline:  {"YES" if result["accuracy"] > 0.65 else "NO"}')

## 6. Voice Pipeline Test

T4 can't hold all three models at once. Load order:
1. Unload LLM (free ~3GB)
2. Load Canary-Qwen STT (~5GB)
3. Reload LLM in 4-bit (~3GB) → total ~8GB
4. Run STT + LLM turn
5. Unload both, load Fish TTS (~5GB 4-bit)
6. Synthesise response

In [ ]:
import torch, gc

# Unload LLM if still in memory
if 'llm' in dir() and llm is not None:
    llm.unload()
    llm = None
    gc.collect()
    torch.cuda.empty_cache()

print(f'VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

In [ ]:
# Load STT (~5GB)
from stt.model import load_canary
canary = load_canary()
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Reload LLM in 4-bit alongside STT (~8GB total)
from llm.client import LLMClient
llm = LLMClient(model_name='Qwen/Qwen3-4B', load_in_4bit=True)
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
# Record 5 seconds via browser mic
from IPython.display import Javascript, Audio, display
from google.colab import output
import base64, os

AUDIO_PATH = '/content/user_input.wav'

RECORD_JS = """
const sleep = ms => new Promise(r => setTimeout(r, ms));
const b64   = b  => new Promise(r => { const fr = new FileReader(); fr.onload = () => r(fr.result); fr.readAsDataURL(b); });
const stream = await navigator.mediaDevices.getUserMedia({audio: true});
const rec = new MediaRecorder(stream);
const chunks = [];
rec.ondataavailable = e => chunks.push(e.data);
rec.start();
await sleep(5000);
rec.stop();
await sleep(300);
const data = await b64(new Blob(chunks));
google.colab.kernel.invokeFunction('notebook.save_audio', [data], {});
"""

def save_audio(data):
    audio_bytes = base64.b64decode(data.split(',')[1])
    with open('/content/raw.webm', 'wb') as f:
        f.write(audio_bytes)
    os.system(f'ffmpeg -i /content/raw.webm -ar 16000 -ac 1 {AUDIO_PATH} -y -loglevel quiet')
    print(f'Saved: {AUDIO_PATH}')

output.register_callback('notebook.save_audio', save_audio)
display(Javascript(RECORD_JS))
print('Recording 5 seconds — speak now...')

In [ ]:
# STT + memory extraction + LLM generation
from stt.transcribe import transcribe_with_embedding
from memory.extractor import extract_turn
from memory.store import HippoMemory
from memory.retriever import hippo_retrieve
from llm.context import build_system_prompt, BASE_COMPANION_PROMPT
import numpy as np

# Transcribe
transcript, audio_emb = transcribe_with_embedding(canary, AUDIO_PATH)
print(f'Transcript:  "{transcript}"')
print(f'Emb std:     {audio_emb.std():.3f}')

# Extract memories
memories = extract_turn(transcript, audio_emb, llm)
print(f'\nExtracted {len(memories)} memories:')
for m in memories:
    print(f'  [{m["emotion"]["label"]} {m["emotion"]["intensity"]:.2f}] {m["content"]}')

# Generate response
system = build_system_prompt([], BASE_COMPANION_PROMPT)
response_text = llm.generate(
    system=system,
    messages=[{'role': 'user', 'content': transcript}],
    max_tokens=100
)
print(f'\nResponse: "{response_text}"')

In [ ]:
# Swap: unload STT + LLM, load TTS
del canary, llm
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.1f} GB')

from tts.model import load_fish_tts
fish = load_fish_tts()
print(f'VRAM used after TTS load: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

In [ ]:
from tts.synthesize import synthesize
OUTPUT_PATH = '/content/response.wav'
synthesize(fish, response_text, OUTPUT_PATH)
display(Audio(OUTPUT_PATH, autoplay=True))
print('Done.')

## 7. Save Results

In [ ]:
import json, datetime
from google.colab import files

out = {
    'timestamp': datetime.datetime.now().isoformat(),
    'gpu': 'T4',
    'llm': 'Qwen3-4B-4bit',
    'signal_noise': {k: v for k, v in hippo_result.items() if k != 'retrieved'} if 'hippo_result' in dir() else 'not run',
    'naive_rag':    {k: v for k, v in naive_result.items()  if k != 'retrieved'} if 'naive_result'  in dir() else 'not run',
    'locomo':       {'accuracy': result['accuracy']} if 'result' in dir() else 'not run',
}

with open('/content/hippovoice_results.json', 'w') as f:
    json.dump(out, f, indent=2)

files.download('/content/hippovoice_results.json')
print(json.dumps(out, indent=2))